# ⚙️ Qdrant Advanced Features Demo

## What You'll Learn

This notebook demonstrates **advanced Qdrant capabilities** through Llama Stack:

| Feature | What It Does | Why You Need It |
|---------|--------------|------------------|
| **Metadata Filtering** | Filter results by document attributes | Narrow results to specific categories, dates, etc. |
| **Score Threshold** | Set minimum relevance score | Ensure only high-quality matches are returned |
| **Chunking Strategies** | Control how documents are split | Optimize for your content type |
| **Search Mode Comparison** | Compare vector/keyword/hybrid | Choose the right mode for your use case |

## Real-World Use Cases

- **E-commerce**: Filter products by category while searching by description
- **Document search**: Filter by date range, author, or department
- **Support tickets**: Filter by priority while searching by content
- **News search**: Filter by source while searching by topic

## Prerequisites

1. ✅ Ollama running with embedding models
2. ✅ Qdrant running on port 6333
3. ✅ Llama Stack running on port 8321 with Qdrant configured:
   ```bash
   OLLAMA_URL=http://localhost:11434/v1 QDRANT_URL=http://localhost:6333 llama stack run starter --port 8321
   ```

---
## Step 1: Connect and Load Data

**What we're doing:** Connecting to Llama Stack and loading the startups dataset.

**Expected:** Connection confirmation and data statistics.

In [ ]:
import json
import io
from pathlib import Path
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8321/v1/",
    api_key="none"
)

# Load startups dataset
data_path = Path("../data/startups_demo.json")
startups_data = []
with open(data_path) as f:
    for line in f:
        if line.strip():
            startups_data.append(json.loads(line))

print(f"✅ Connected to Llama Stack!")
print(f"📊 Loaded {len(startups_data):,} startups")

---
## Step 2: Create Vector Store

**What we're doing:** Creating a vector store for our demo.

**Expected:** Vector store ID.

In [ ]:
vector_store = client.vector_stores.create(
    name="advanced_features_demo",
    extra_body={
        "provider_id": "qdrant",
        "embedding_model": "ollama/nomic-embed-text:latest"
    }
)

print(f"✅ Created vector store: {vector_store.id}")

---
## Step 3: Insert Documents WITH Metadata

**What we're doing:** Inserting startup documents with rich metadata attributes:
- `city`: Where the company is located
- `category`: Type of business (software, healthcare, fintech, other)
- `index`: Document number (for range filtering demos)
- `name`: Company name

**Why metadata matters:**
- Enables filtering without re-embedding
- Combine semantic search with structured queries
- Much faster than searching then filtering

**Expected:** Progress as 25 documents are inserted with metadata.

In [ ]:
# Use 25 startups for demo
demo_startups = startups_data[:25]
print(f"📥 Inserting {len(demo_startups)} documents with metadata...\n")

for i, startup in enumerate(demo_startups):
    city = startup.get('city', 'Unknown')
    name = startup.get('name', 'Unknown')
    
    # Categorize based on description keywords
    desc = startup.get('description', '').lower()
    if 'software' in desc or 'platform' in desc or 'app' in desc:
        category = 'software'
    elif 'health' in desc or 'medical' in desc or 'patient' in desc:
        category = 'healthcare'
    elif 'finance' in desc or 'payment' in desc or 'money' in desc:
        category = 'fintech'
    else:
        category = 'other'
    
    content = f"""Company: {name}
City: {city}
Category: {category}
Description: {startup.get('description', 'No description')}
"""
    
    pseudo_file = io.BytesIO(content.encode("utf-8"))
    uploaded_file = client.files.create(
        file=(f"startup_{i}.txt", pseudo_file, "text/plain"),
        purpose="assistants"
    )
    
    # IMPORTANT: Store metadata attributes with the file
    # These attributes can be used for filtering later!
    client.vector_stores.files.create(
        vector_store_id=vector_store.id,
        file_id=uploaded_file.id,
        extra_body={
            "attributes": {
                "city": city,
                "name": name,
                "category": category,
                "index": i  # Numeric attribute for range queries
            }
        }
    )
    
    if (i + 1) % 5 == 0:
        print(f"   ✓ {i + 1}/{len(demo_startups)} documents inserted")

print(f"\n✅ All documents inserted with metadata!")
print(f"   Each document has: city, name, category, index attributes")

---
# Part 1: Metadata Filtering

## What is Metadata Filtering?

Metadata filtering lets you **narrow search results** based on document attributes, without affecting semantic search.

```
Query: "innovative technology"
Filter: category = "software"
              ↓
Result: Only software companies that match "innovative technology"
```

## Available Filter Types

| Filter | Meaning | Example |
|--------|---------|--------|
| `eq` | Equal to | `{"type": "eq", "key": "city", "value": "Chicago"}` |
| `ne` | Not equal | `{"type": "ne", "key": "category", "value": "other"}` |
| `gt` | Greater than | `{"type": "gt", "key": "price", "value": 100}` |
| `gte` | Greater or equal | `{"type": "gte", "key": "rating", "value": 4.0}` |
| `lt` | Less than | `{"type": "lt", "key": "age", "value": 30}` |
| `lte` | Less or equal | `{"type": "lte", "key": "count", "value": 50}` |
| `and` | All must match | Combine multiple filters |
| `or` | Any must match | Match any of multiple filters |

In [ ]:
# Helper function for filtered search
def search_with_filter(query, filters=None, max_results=5):
    """Search with optional metadata filters."""
    results = client.vector_stores.search(
        vector_store_id=vector_store.id,
        query=query,
        max_num_results=max_results,
        filters=filters,
        extra_body={"search_mode": "vector"}
    )
    return results

def display_with_metadata(results, title):
    """Display results showing metadata attributes."""
    print(f"\n{'='*70}")
    print(f"🔍 {title}")
    print(f"{'='*70}")
    
    if not results.data:
        print("   ❌ No results match the filter")
        return
    
    for i, r in enumerate(results.data, 1):
        attrs = r.attributes or {}
        idx = attrs.get('index', 'N/A')
        idx_display = int(idx) if isinstance(idx, float) else idx
        print(f"\n   {i}. {attrs.get('name', 'Unknown')}")
        print(f"      Category: {attrs.get('category', 'N/A')}")
        print(f"      City: {attrs.get('city', 'N/A')}")
        print(f"      Index: {idx_display}")
        print(f"      Score: {r.score:.4f}")

print("✅ Helper functions defined!")

### Step 4a: Baseline - No Filters

**What we're doing:** First, let's see all results without any filters.

**Why:** This establishes a baseline to compare against filtered results.

In [ ]:
results = search_with_filter("technology company", filters=None)
display_with_metadata(results, "BASELINE: No Filters (all results)")

print("\n💡 Notice the mix of categories and index values.")
print("   Now let's filter to see only specific subsets!")

### Step 4b: Equality Filter (eq)

**What we're doing:** Filter to show ONLY software companies.

**Why:** When you know exactly what category you want.

In [ ]:
# Filter: category = "software"
results = search_with_filter(
    "technology company",
    filters={"type": "eq", "key": "category", "value": "software"}
)
display_with_metadata(results, "FILTER: category = 'software'")

print("\n💡 All results now have category = 'software'")

### Step 4c: Not Equal Filter (ne)

**What we're doing:** Exclude 'other' category - show everything BUT 'other'.

**Why:** When you want to exclude unclassified or unwanted items.

In [ ]:
# Filter: category != "other"
results = search_with_filter(
    "technology company",
    filters={"type": "ne", "key": "category", "value": "other"}
)
display_with_metadata(results, "FILTER: category != 'other'")

print("\n💡 Results include software, healthcare, fintech - but NOT 'other'")

### Step 4d: Range Filters (gt, lt, gte, lte)

**What we're doing:** Filter by numeric ranges using the index attribute.

**Why:** Perfect for dates, prices, ratings, counts, etc.

In [ ]:
# Filter: index > 15 (only documents 16-24)
results = search_with_filter(
    "technology company",
    filters={"type": "gt", "key": "index", "value": 15}
)
display_with_metadata(results, "FILTER: index > 15")

print("\n💡 All results have index greater than 15")

In [ ]:
# Filter: index <= 5 (only documents 0-5)
results = search_with_filter(
    "technology company",
    filters={"type": "lte", "key": "index", "value": 5}
)
display_with_metadata(results, "FILTER: index <= 5")

print("\n💡 All results have index 5 or less")

### Step 4e: Combined Filters (AND)

**What we're doing:** Combine multiple conditions where ALL must be true.

**Why:** Complex queries like "software companies created recently" or "healthcare in specific region".

In [ ]:
# Filter: category != 'other' AND index <= 15
results = search_with_filter(
    "technology",
    filters={
        "type": "and",
        "filters": [
            {"type": "ne", "key": "category", "value": "other"},
            {"type": "lte", "key": "index", "value": 15}
        ]
    }
)
display_with_metadata(results, "FILTER: category != 'other' AND index <= 15")

print("\n💡 Results must satisfy BOTH conditions")

### Step 4f: Combined Filters (OR)

**What we're doing:** Match ANY of multiple conditions.

**Why:** When you want items from multiple categories, e.g., "software OR healthcare".

In [ ]:
# Filter: category = 'software' OR category = 'healthcare'
results = search_with_filter(
    "company",
    filters={
        "type": "or",
        "filters": [
            {"type": "eq", "key": "category", "value": "software"},
            {"type": "eq", "key": "category", "value": "healthcare"}
        ]
    }
)
display_with_metadata(results, "FILTER: category = 'software' OR category = 'healthcare'")

print("\n💡 Results are EITHER software OR healthcare (not fintech or other)")

---
### Step 4g: Filters with Different Search Modes

**What we're doing:** Combining metadata filters with keyword and hybrid search modes.

**Why:** Filters work with all three search modes -- not just vector search.

In [ ]:
# Keyword search with category filter
query = "platform"
software_filter = {"type": "eq", "key": "category", "value": "software"}

print(f"📝 Query: '{query}' | Filter: category = 'software'\n")

for mode in ["vector", "keyword", "hybrid"]:
    results = client.vector_stores.search(
        vector_store_id=vector_store.id,
        query=query,
        max_num_results=3,
        filters=software_filter,
        extra_body={"search_mode": mode}
    )
    display_with_metadata(results, f"{mode.upper()} + Filter")

print("\n💡 All three modes respect the metadata filter.")
print("   • Vector: ranked by semantic similarity")
print("   • Keyword: all scores 1.0 (literal match)")
print("   • Hybrid: vector similarity filtered by keyword match")

---
# Part 2: Score Threshold

## What is Score Threshold?

Score threshold sets a **minimum relevance score** for results. Results below this threshold are not returned.

```
Search results:       With threshold=0.5:
  0.8 - Match A       0.8 - Match A  ✓
  0.6 - Match B       0.6 - Match B  ✓
  0.4 - Match C       (filtered out) ✗
  0.2 - Match D       (filtered out) ✗
```

## Why Use Score Threshold?

- **Quality control**: Only return highly relevant results
- **User experience**: Avoid showing poor matches
- **Performance**: Return fewer, better results

In [ ]:
def search_with_threshold(query, threshold, max_results=10):
    """Search with a minimum score threshold."""
    results = client.vector_stores.search(
        vector_store_id=vector_store.id,
        query=query,
        max_num_results=max_results,
        ranking_options={"score_threshold": threshold},
        extra_body={"search_mode": "vector"}
    )
    return results

print("✅ Threshold search function defined!")

In [ ]:
# Compare different thresholds
query = "machine learning artificial intelligence"

print(f"📝 Query: '{query}'")
print(f"\n{'='*70}")
print("COMPARING SCORE THRESHOLDS")
print(f"{'='*70}")

for threshold in [0.0, 0.3, 0.5, 0.7]:
    results = search_with_threshold(query, threshold)
    count = len(results.data)
    
    if count > 0:
        scores = [r.score for r in results.data]
        print(f"\n   Threshold >= {threshold}:")
        print(f"   • Results: {count}")
        print(f"   • Score range: {min(scores):.3f} - {max(scores):.3f}")
    else:
        print(f"\n   Threshold >= {threshold}: 0 results (too strict)")

print("\n💡 Higher threshold = fewer but more relevant results")

In [ ]:
# Show actual results at different thresholds
print("\n--- Low Threshold (0.0) - All results ---")
results = search_with_threshold(query, 0.0, max_results=5)
for i, r in enumerate(results.data, 1):
    attrs = r.attributes or {}
    name = attrs.get('name', 'Unknown')
    cat = attrs.get('category', '?')
    print(f"   {i}. Score: {r.score:.4f} | {name} ({cat})")

print("\n--- High Threshold (0.5) - Quality filter ---")
results = search_with_threshold(query, 0.5, max_results=5)
if results.data:
    for i, r in enumerate(results.data, 1):
        attrs = r.attributes or {}
        name = attrs.get('name', 'Unknown')
        cat = attrs.get('category', '?')
        print(f"   {i}. Score: {r.score:.4f} | {name} ({cat})")
else:
    print("   No results meet the threshold")

print("\n💡 With threshold 0.5, only highly relevant results are shown")

---
# Part 3: Chunking Strategies

## What is Chunking?

When you upload a document, it's split into **chunks** before embedding. Chunking strategy controls:
- **Chunk size**: How many tokens per chunk
- **Overlap**: How much chunks overlap (for context continuity)

```
Original document: "The quick brown fox jumps over the lazy dog..."
                            ↓ Chunking
Chunk 1: "The quick brown fox jumps"
Chunk 2: "fox jumps over the lazy"  ← overlap!
Chunk 3: "the lazy dog..."
```

## Available Strategies

| Strategy | Description |
|----------|-------------|
| **auto** | Default: 800 tokens, 400 overlap |
| **static** | Custom size and overlap |

In [ ]:
# Create vector store with CUSTOM chunking
vector_store_custom = client.vector_stores.create(
    name="custom_chunking_demo",
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 150,   # Smaller chunks
            "chunk_overlap_tokens": 30      # Less overlap
        }
    },
    extra_body={
        "provider_id": "qdrant",
        "embedding_model": "ollama/nomic-embed-text:latest"
    }
)

print(f"✅ Created vector store with custom chunking:")
print(f"   ID: {vector_store_custom.id}")
print(f"   Max chunk size: 150 tokens (vs default 800)")
print(f"   Chunk overlap: 30 tokens (vs default 400)")

In [ ]:
# Insert a longer document to see chunking in action
# This document is deliberately long to produce multiple chunks at 150 tokens
long_content = """Artificial Intelligence Company Profile

Chapter 1: Company Overview

Our company specializes in developing cutting-edge artificial intelligence solutions
for enterprise customers. We focus on machine learning, natural language processing,
and computer vision technologies. Founded in San Francisco in 2020, we have grown
rapidly to become one of the most innovative AI startups in the Bay Area. Our mission
is to democratize artificial intelligence and make it accessible to businesses of all
sizes, from small startups to Fortune 500 companies. We believe that AI should be a
tool that augments human capabilities rather than replacing them.

Chapter 2: Products and Services

Our flagship product, IntelliSense Platform, uses deep learning algorithms to analyze
customer behavior patterns and predict future trends. The platform integrates seamlessly
with existing business intelligence tools including Tableau, Power BI, and Looker.
We also offer a suite of natural language processing tools that can extract insights
from unstructured text data, including customer reviews, support tickets, and social
media posts. Our computer vision module can process images and video feeds in real-time,
enabling applications in manufacturing quality control, retail analytics, and security.

Chapter 3: Technology Stack

Our infrastructure runs on a distributed computing framework that can scale horizontally
to handle petabytes of data. We use a combination of TensorFlow, PyTorch, and our
proprietary ML framework called NeuralForge. Our models are trained on custom GPU clusters
with thousands of NVIDIA A100 GPUs, and we use advanced techniques like federated learning,
model distillation, and neural architecture search to optimize performance. We also
maintain a comprehensive MLOps pipeline with automated model training, evaluation,
versioning, and deployment capabilities.

Chapter 4: Team and Culture

Our team consists of over 200 PhD researchers and industry veterans with decades of
combined experience in AI and data science. We have assembled talent from leading
institutions including Stanford, MIT, Carnegie Mellon, and Google Brain. Our culture
emphasizes innovation, collaboration, and continuous learning. Every engineer participates
in weekly paper reading groups and has a 20% time allocation for personal research projects.
We publish regularly at top conferences like NeurIPS, ICML, and CVPR.

Chapter 5: Clients and Impact

We serve over 500 enterprise clients worldwide across industries including healthcare,
finance, retail, and manufacturing. In healthcare, our diagnostic AI has helped reduce
misdiagnosis rates by 30%. In finance, our fraud detection system processes over 10 million
transactions per day with a 99.7% accuracy rate. Our retail analytics platform has helped
clients increase revenue by an average of 15% through personalized recommendations.
"""

pseudo_file = io.BytesIO(long_content.encode("utf-8"))
uploaded_file = client.files.create(
    file=("ai_company.txt", pseudo_file, "text/plain"),
    purpose="assistants"
)

client.vector_stores.files.create(
    vector_store_id=vector_store_custom.id,
    file_id=uploaded_file.id
)

print(f"✅ Inserted document (~{len(long_content.split())} words)")
print(f"   With max_chunk_size=150 tokens, this will produce multiple chunks")

In [ ]:
# Search different topics to see chunks from different sections
queries = [
    "machine learning enterprise platform",
    "team culture researchers",
    "healthcare fraud detection clients"
]

for query in queries:
    results = client.vector_stores.search(
        vector_store_id=vector_store_custom.id,
        query=query,
        max_num_results=3,
        extra_body={"search_mode": "vector"}
    )
    
    print(f"\n{'='*70}")
    print(f"🔍 Query: '{query}' — Found {len(results.data)} chunks")
    print(f"{'='*70}")
    
    for i, r in enumerate(results.data, 1):
        text = r.content[0].text if r.content else "N/A"
        preview = text[:150].replace('\n', ' ') + "..."
        print(f"\n   Chunk {i} (Score: {r.score:.4f}):")
        print(f"   '{preview}'")

print("\n💡 Smaller chunks = more granular search results!")
print("   Each query finds chunks from the specific section of the document.")
print("   'team culture' finds Chapter 4, 'healthcare fraud' finds Chapter 5, etc.")

---
## Cleanup

**What we're doing:** Removing test vector stores.

In [ ]:
# Clean up both vector stores
client.vector_stores.delete(vector_store.id)
print(f"🗑️  Deleted: {vector_store.id}")

client.vector_stores.delete(vector_store_custom.id)
print(f"🗑️  Deleted: {vector_store_custom.id}")

print(f"\n✅ Advanced features demo complete!")

---
## 📚 Summary

### What We Learned

| Feature | What It Does | When to Use |
|---------|--------------|-------------|
| **Metadata Filtering** | Filter by document attributes (eq, ne, gt, lt, and, or) | Narrowing by category, date, author, etc. |
| **Filters + Search Modes** | Combine filters with vector, keyword, or hybrid search | Full flexibility for any query pattern |
| **Score Threshold** | Set minimum relevance score via `ranking_options` | Quality control, fewer but better results |
| **Chunking Strategy** | Control document splitting (auto vs static) | Optimize for your content type and granularity |

### Filter Types Reference

```python
# Comparison filters
{"type": "eq", "key": "status", "value": "active"}      # Equal
{"type": "ne", "key": "status", "value": "deleted"}     # Not equal
{"type": "gt", "key": "price", "value": 100}            # Greater than
{"type": "gte", "key": "price", "value": 100}           # Greater or equal
{"type": "lt", "key": "count", "value": 50}             # Less than
{"type": "lte", "key": "count", "value": 50}            # Less or equal

# Logical filters
{"type": "and", "filters": [filter1, filter2]}  # All must match
{"type": "or", "filters": [filter1, filter2]}   # Any must match

# Simple key-value (shorthand for eq)
{"topic": "ai", "category": "tech"}
```

### Key Takeaways

1. **Always include metadata** when inserting documents -- you'll thank yourself later
2. **Filters work with all search modes** -- vector, keyword, and hybrid
3. **Use score threshold** in production to ensure quality results
4. **Adjust chunking** based on your content -- smaller for precise search, larger for context
5. **Combine features** -- e.g., hybrid search + metadata filter + score threshold for production use